# 03 Human check of the claim checker

Does the deterministic checker agree with people? Two annotators look at the before and after crops of one cell and judge one claim, without seeing the checker's verdict.

**Phase 1 (run now):** pool claims, draw a sample stratified by checker verdict, write crops and the annotator sheets.
**Phase 2 (run after both annotators finish):** agreement between annotators (raw agreement, Cohen's kappa), consensus labels, checker against human confusion matrix.

Runs on CPU. It only reads files saved by notebook 02, so no model is loaded. Clear outputs before committing. Never commit the crops, sheets or key file (they stay on Drive).

In [ ]:
BRANCH = "main"

import os, subprocess, sys

REPO_DIR = "/content/FloodVerify"
if "google.colab" in sys.modules:
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "https://github.com/aaarrvind/FloodVerify.git", REPO_DIR], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "pillow", "matplotlib"], check=True)
    REPO = REPO_DIR
else:
    REPO = os.path.abspath("../..")
sys.path.insert(0, os.path.join(REPO, "src"))
print("repo:", REPO)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

ROOT = "/content/drive/MyDrive/MAAI 902 X 901 PROJECT/xbd"
TILES = f"{ROOT}/processed/xbd"
EVIDENCE_DIR = f"{ROOT}/processed/evidence_labels"
CAPTIONS = f"{ROOT}/processed/captions"
HUMAN_DIR = f"{CAPTIONS}/human_check"            # sheets, crops and the private key live here
os.makedirs(HUMAN_DIR, exist_ok=True)

MAIN_MODEL_NAME = "Qwen3-VL-4B (50 tiles)"       # value of the model column in claims_verdicts.csv
RSCC_CLAIMS = f"{CAPTIONS}/rscc_claims_verdicts.csv"   # optional; see the RSCC cell below
SEED = 42
LABELS = ["correct", "incorrect", "region not visible"]

## Load the claims from notebook 02

In [ ]:
import json, random
import numpy as np
import pandas as pd

claims_all = pd.read_csv(f"{CAPTIONS}/claims_verdicts.csv")
tab_main = claims_all[claims_all["model"] == MAIN_MODEL_NAME].reset_index(drop=True)
print(len(tab_main), "Qwen3-VL claims")
print(tab_main["verdict"].value_counts())

## Claims from the RSCC captions

`rscc_flood_manifest.csv` (in the `xbd` folder) holds one RSCC caption per 512-pixel part of an xBD image. A row maps to your tile as `<event>_<tile_id>_p<part>`.

RSCC captions are free text about the whole part, not per cell, so the notebook keeps only the clearest statements: captions that say **all (or both, or the single) buildings are undamaged** become the claim "none damaged", and captions that say **all buildings have major damage, severe damage or are destroyed** become "most damaged". Captions that mention unknown or obscured buildings, partly damaged buildings, or no buildings are skipped. About 1 in 5 captions gives a claim.

Each claim is about the whole part. The checker's verdict uses all usable cells of the tile together: the damaged share is the building-weighted share over cells with buildings and at least half real imagery, and the claim is supported when its quantity word equals the evidence bin (the same exact-bin rule as the cell-level checker). Report this as a tile-level variant of the check.

In [ ]:
import re

RSCC_FILE = f"{ROOT}/rscc_flood_manifest.csv"

manifest = pd.read_csv(RSCC_FILE, dtype={"tile_id": str, "part": str})
manifest["tile"] = manifest["event"] + "_" + manifest["tile_id"] + "_p" + manifest["part"]
have = {f[:-5] for f in os.listdir(EVIDENCE_DIR) if f.endswith(".json")}
print(manifest.shape, list(manifest.columns))
print("captions with a matching evidence file:", int(manifest["tile"].isin(have).sum()), "of", len(manifest))
print(manifest["event"].value_counts())

In [ ]:
BUILD = r"(?:buildings?|structures?|houses?|homes?)"
ALL = rf"(?:\b(?:all|both|every)\b[^.;]{{0,40}}\b{BUILD}\b|\b(?:single|solitary|sole|lone)\s+{BUILD}\b)"
UNDAM = (r"(?:\bundamaged\b|\bno damage\b|\bno visible damage\b|\bno structural damage\b|\blevel 0\b|\bunaffected\b|"
         r"\bundisturbed\b|\bremain(?:s|ed)?\s+(?:structurally\s+)?intact\b|\bretain(?:s|ed|ing)?\b[^.;]{0,30}\bintegrity\b)")
DAM = r"(?:\bmajor damage\b|\blevel 2\b|\bsevere(?:ly)? damage[d]?\b|\bdestroyed\b|\blevel 3\b)"
UNKNOWN = r"\b(?:unclassified|unknown|unassess\w*|obscured|unclear|undetermined|cannot be assessed|impossible|inconclusive)\b"
NO_BUILDINGS = rf"\bno {BUILD}\b[^.;]{{0,40}}\b(?:present|exist|visible)\b|\bno {BUILD} (?:are|were) present\b"
PARTIAL = rf"\b(?:one|two|three|four|five|six|seven|eight|nine|ten|\d+)\s+(?:of\s+\w+\s+)?{BUILD}\b[^.;]{{0,60}}\b(?:major|minor|destroyed|level [123])\b"


def rscc_tile_claim(caption):
    """'none' or 'most' when the caption clearly says that all or none of the buildings are damaged, else None."""
    c = caption.lower()
    if re.search(NO_BUILDINGS, c) or re.search(UNKNOWN, c):
        return None
    sents = re.split(r"[.;]", c)
    none = any(re.search(ALL, s) and re.search(UNDAM, s) and not re.search(DAM, s) for s in sents)
    most = any(re.search(ALL, s) and re.search(DAM, s) and not re.search(UNDAM, s) for s in sents)
    if none == most:
        return None
    if none and re.search(PARTIAL, c):
        return None
    return "none" if none else "most"


rscc_claims = manifest.assign(quantity=manifest["caption"].map(rscc_tile_claim)).dropna(subset=["quantity"])
print(len(rscc_claims), "claims from", len(manifest), "captions")
print(rscc_claims["quantity"].value_counts())

In [ ]:
def tile_verdict(quantity, evidence):
    """Verdict for a whole-tile claim about the share of damaged buildings, using all usable cells together."""
    from floodsitrep.nlp.checker import bin_of, class_fraction
    usable = [c for c in evidence["cells"] if c["buildings"] > 0 and c.get("valid_fraction", 1.0) >= 0.5]
    total = sum(c["buildings"] for c in usable)
    if total == 0:
        return "not_verifiable"
    damaged = sum(c["buildings"] * (class_fraction([c], "damaged") or 0.0) for c in usable)
    return "supported" if bin_of(damaged / total) == quantity else "contradicted"


def rscc_claims_table(claims, evidence_dir):
    """Pool-format table for the RSCC tile-level claims that have an evidence file."""
    rows = []
    for _, r in claims.iterrows():
        path = f"{evidence_dir}/{r['tile']}.json"
        if not os.path.exists(path):
            continue
        rows.append({"tile_id": r["tile"], "cell": "all", "quantity": r["quantity"], "damage_class": "damaged",
                     "verdict": tile_verdict(r["quantity"], json.load(open(path)))})
    return pd.DataFrame(rows, columns=["tile_id", "cell", "quantity", "damage_class", "verdict"])


rscc_tab = rscc_claims_table(rscc_claims, EVIDENCE_DIR)
print(len(rscc_tab), "RSCC claims with evidence")
print(pd.crosstab(rscc_tab["quantity"], rscc_tab["verdict"]))
rscc_tab.to_csv(RSCC_CLAIMS, index=False)

## Reference claims built from the evidence

The RSCC claims are about whole images and use only the words none and most. To add cell-level claims with every quantity word, this section builds reference claims from the evidence itself: for a random usable cell (buildings present, at least half real imagery) it writes either the true quantity word or a word at least two bins away, about "damaged" (any damage level). The checker gives each one its verdict as usual.

These claims are not model output. They test whether the checker's verdict, which rests on the evidence labels, agrees with what people see in the pictures. The model's own claims (Qwen3-VL) stay in the pool, and every result is reported per source. Say in the report that this source exists and how it was built.

In [ ]:
BINS = ["none", "few", "some", "many", "most"]


def reference_claims(tile_ids, evidence_dir, per_tile=3, seed=SEED):
    """Claims built from the evidence: per usable cell, the true quantity word or one two or more bins away."""
    from floodsitrep.nlp.checker import bin_of, class_fraction, verify_claims
    from floodsitrep.nlp.schema import Claim
    rng = random.Random(seed)
    rows = []
    for tid in tile_ids:
        ev = json.load(open(f"{evidence_dir}/{tid}.json"))
        usable = [c for c in ev["cells"] if c["buildings"] > 0 and c.get("valid_fraction", 1.0) >= 0.5]
        for c in rng.sample(usable, min(per_tile, len(usable))):
            true_q = bin_of(class_fraction([c], "damaged"))
            far = [b for b in BINS if abs(BINS.index(b) - BINS.index(true_q)) >= 2]
            q = rng.choice([true_q, rng.choice(far)])
            claim = Claim(text=f"{q} buildings in {c['cell']} are damaged", cell=c["cell"],
                          quantity=q, damage_class="damaged")
            verdict = verify_claims([claim], ev)[0].verdict
            rows.append({"tile_id": tid, "cell": c["cell"], "quantity": q, "damage_class": "damaged", "verdict": verdict})
    return pd.DataFrame(rows, columns=["tile_id", "cell", "quantity", "damage_class", "verdict"])


ref_tab = reference_claims(sorted(tab_main["tile_id"].unique()), EVIDENCE_DIR)
print(len(ref_tab), "reference claims")
print(ref_tab["verdict"].value_counts())

## Phase 1: pool, sample, crops, sheets

In [ ]:
POOL_COLUMNS = ["source", "tile_id", "cell", "quantity", "damage_class", "verdict"]


def build_pool(tables):
    """Pool claims from several sources (a dict of source name to claims table); skips sources that are None."""
    parts = [t.assign(source=name)[POOL_COLUMNS] for name, t in tables.items() if t is not None]
    return pd.concat(parts, ignore_index=True)


def claim_text(row):
    """Plain wording of the claim shown to annotators (no verdict, no evidence numbers)."""
    q, cls = row["quantity"], row["damage_class"]
    if row["cell"] == "all":
        return f"{q} of the buildings in the whole image are damaged"
    if cls == "damaged":
        return f"{q} of the buildings in this area are damaged"
    return f"{q} of the buildings in this area show {cls} damage"


def sample_claims(pool, quotas, seed=SEED):
    """Sample per (source, verdict). quotas maps (source, verdict) to a count; takes all if fewer are available."""
    rng = random.Random(seed)
    parts = []
    for (source, verdict), n in quotas.items():
        ids = list(pool.index[(pool["source"] == source) & (pool["verdict"] == verdict)])
        parts.append(pool.loc[rng.sample(ids, min(n, len(ids)))])
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

In [ ]:
def has_images(tiles_dir, tile_id):
    """True when both the before and after picture of the tile are in tiles_dir."""
    return all(os.path.exists(f"{tiles_dir}/{tile_id}/{n}.png") for n in ("pre", "post"))


def keep_with_images(pool, tiles_dir):
    """Drop claims whose tile has no pictures in tiles_dir, and say how many were dropped."""
    ok = {t: has_images(tiles_dir, t) for t in pool["tile_id"].unique()}
    keep = pool[pool["tile_id"].map(ok)].reset_index(drop=True)
    print(f"kept {len(keep)} of {len(pool)} claims whose tile pictures are in TILES ({sum(ok.values())} of {len(ok)} tiles)")
    return keep


pool = build_pool({"qwen3vl": tab_main, "rscc": rscc_tab, "evidence_ref": ref_tab})
pool = keep_with_images(pool, TILES)
print(pool.groupby(["source", "verdict"]).size())

In [ ]:
from PIL import Image, ImageDraw

GRID = 3


def cell_box(cell, width, height, grid=GRID):
    """Pixel box (left, top, right, bottom) of a cell named like r1c2 (row 1, column 2)."""
    if cell == "all":
        return (0, 0, width, height)
    r, c = int(cell[1]), int(cell[3])
    w, h = width // grid, height // grid
    return (c * w, r * h, (c + 1) * w, (r + 1) * h)


def make_crop(tiles_dir, tile_id, cell, out_path, scale=3):
    """Side-by-side before and after crop of one cell, enlarged so people can see it (whole images at 2x, cells at 3x)."""
    pre = Image.open(f"{tiles_dir}/{tile_id}/pre.png").convert("RGB")
    post = Image.open(f"{tiles_dir}/{tile_id}/post.png").convert("RGB")
    box = cell_box(cell, *pre.size)
    scale = 2 if cell == "all" else scale
    a, b = pre.crop(box), post.crop(box)
    a = a.resize((a.width * scale, a.height * scale), Image.LANCZOS)
    b = b.resize((b.width * scale, b.height * scale), Image.LANCZOS)
    canvas = Image.new("RGB", (a.width + b.width + 12, a.height + 22), "white")
    canvas.paste(a, (0, 22))
    canvas.paste(b, (a.width + 12, 22))
    d = ImageDraw.Draw(canvas)
    d.text((4, 4), "BEFORE", fill="black")
    d.text((a.width + 16, 4), "AFTER", fill="black")
    canvas.save(out_path)


GUIDE = """HOW TO LABEL
Each row shows a BEFORE and AFTER picture of one small area and one statement about the buildings in it.
Write one label in the label column:
  correct             the statement fits what the pictures show
  incorrect           it does not fit
  region not visible  you cannot see the buildings at all: cloud, haze, a missing image, or the area is outside the picture

If you can see the buildings but are unsure whether they are damaged, do not use "region not visible".
Choose correct or incorrect as your best judgement and add a short note in the note column (for example: unsure, blurry).

What the words mean (share of the buildings in the area):
  none  = no building is damaged
  few   = fewer than about 1 in 10
  some  = about 1 in 10 up to about 3 in 10
  many  = about 3 in 10 up to about 6 in 10
  most  = more than about 6 in 10
Judge the share of the buildings, not how many there are. Example: if the area holds 3 buildings and all 3 are damaged,
"few of the buildings are damaged" is incorrect, even though 3 is a small number.
"damaged" means any damage (minor, major or destroyed). "minor", "major" and "destroyed" name the level of damage.
Some rows show a whole image instead of one small area. The statement then refers to the whole image.
Judge only what you can see from above. Work alone and do not discuss the rows until both of you have finished.
"""


def write_annotation_files(sample, tiles_dir, out_dir):
    """Crops, a blind annotator sheet (no verdict, no source) and a private key file."""
    os.makedirs(f"{out_dir}/crops", exist_ok=True)
    s = sample.copy()
    s["claim_id"] = range(1, len(s) + 1)
    s["claim"] = s.apply(claim_text, axis=1)
    s["image"] = s["claim_id"].map(lambda i: f"crops/claim_{i:03d}.png")
    for _, r in s.iterrows():
        make_crop(tiles_dir, r["tile_id"], r["cell"], f"{out_dir}/{r['image']}")
    sheet = s[["claim_id", "image", "claim"]].assign(label="", note="")
    sheet.to_csv(f"{out_dir}/annotator_sheet.csv", index=False)
    open(f"{out_dir}/GUIDE.txt", "w").write(GUIDE)
    s[["claim_id", "tile_id", "cell", "source", "verdict", "claim"]].to_csv(f"{out_dir}/key_private.csv", index=False)
    return sheet

### Dry run (about 20 claims)

Run this first and label the 20 claims yourself (next section). Fix any wording or crop that confuses you before the real round.

Label guide for the annotators:
- `correct`: the claim matches what the before and after crops show for this area
- `incorrect`: it does not match
- `region not visible`: cloud, haze or missing image hides the area, so the claim cannot be judged

In [ ]:
DRY_QUOTAS = {("qwen3vl", "supported"): 2, ("qwen3vl", "contradicted"): 3, ("qwen3vl", "not_verifiable"): 2,
              ("evidence_ref", "supported"): 3, ("evidence_ref", "contradicted"): 3,
              ("rscc", "supported"): 3, ("rscc", "contradicted"): 3, ("rscc", "not_verifiable"): 1}
dry = sample_claims(pool, DRY_QUOTAS)
dry_sheet = write_annotation_files(dry, TILES, f"{HUMAN_DIR}/dry_run")
print(len(dry_sheet), "claims written to", f"{HUMAN_DIR}/dry_run")
dry_sheet.head()

### Label the dry run yourself

Do this before anyone else sees the sheet, the way an annotator would: read the guide, then label each row from the pictures and the statement only. Do not open `key_private.csv`. It takes 10 to 15 minutes. Add a note after the letter for any row where you hesitated (for example `i cloud on the left`).

While you work, check three things: the cell crops show the area the statement is about; the "whole image" rows can be judged at that size; you can tell `incorrect` from `region not visible`.

In [ ]:
DRY = f"{HUMAN_DIR}/dry_run"
print(open(f"{DRY}/GUIDE.txt").read())

In [ ]:
from IPython.display import display, clear_output

SHORT = {"c": "correct", "i": "incorrect", "n": "region not visible"}


def label_rows(round_dir, out_name="annotator_dry.csv"):
    """Label the sheet row by row in the notebook. Saves after every answer; rerun to continue."""
    sheet = pd.read_csv(f"{round_dir}/annotator_sheet.csv")
    out_path = f"{round_dir}/{out_name}"
    done = pd.read_csv(out_path) if os.path.exists(out_path) else sheet.copy()
    done["label"] = done["label"].astype("object")
    done["note"] = done["note"].astype("object")
    for i, row in done.iterrows():
        if isinstance(row["label"], str) and row["label"]:
            continue
        while True:
            clear_output(wait=True)
            print(f"row {i + 1} of {len(done)}")
            display(Image.open(f"{round_dir}/{row['image']}"))
            print("STATEMENT:", row["claim"])
            ans = input("c = correct | i = incorrect | n = region not visible | q = stop   (add a note after a space): ").strip()
            if ans[:1].lower() == "q":
                done.to_csv(out_path, index=False)
                return done
            if ans[:1].lower() in SHORT:
                done.at[i, "label"] = SHORT[ans[0].lower()]
                done.at[i, "note"] = ans[1:].strip()
                done.to_csv(out_path, index=False)
                break
    clear_output(wait=True)
    print("all rows labelled, saved to", out_path)
    return done


mine = label_rows(DRY)

Functions for reading labels and comparing them (also used in phase 2).

In [ ]:
def read_labels(path, labels=LABELS):
    """Annotator file as a Series of label by claim_id. Stops on a missing or unknown label."""
    df = pd.read_csv(path).set_index("claim_id")
    bad = df.loc[~df["label"].isin(labels), "label"]
    if len(bad):
        raise ValueError(f"{path}: {len(bad)} rows with a missing or unknown label, first: {bad.head(3).to_dict()}")
    return df["label"]


def agreement_rate(a, b):
    """Share of claims on which two annotators gave the same label."""
    return float((a == b).mean())


def cohen_kappa(a, b, labels=LABELS):
    """Cohen's kappa between two label series."""
    po = agreement_rate(a, b)
    pe = sum((a == k).mean() * (b == k).mean() for k in labels)
    return float((po - pe) / (1 - pe)) if pe < 1 else 1.0


def consensus_labels(a, b):
    """Label both annotators agree on; claims they disagree on get 'disagree' and are listed for adjudication."""
    return a.where(a == b, "disagree")

In [ ]:
def checker_confusion(key, consensus):
    """Checker verdict (rows) against the human consensus label (columns)."""
    k = key.set_index("claim_id").join(consensus.rename("human"))
    return pd.crosstab(k["verdict"], k["human"]), k


def plot_confusion(table, title="Checker verdict against human label"):
    """Heatmap of the confusion table with counts in each cell."""
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6, 3.2))
    im = ax.imshow(table.values, cmap="Blues")
    ax.set_xticks(range(table.shape[1]), table.columns, rotation=20, ha="right")
    ax.set_yticks(range(table.shape[0]), table.index)
    for i in range(table.shape[0]):
        for j in range(table.shape[1]):
            ax.text(j, i, int(table.values[i, j]), ha="center", va="center")
    ax.set_xlabel("human label")
    ax.set_ylabel("checker verdict")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.04)
    fig.tight_layout()
    return fig

After you have labelled all 20 rows, compare your labels with the checker. This is only a check that nothing is badly off. One person and 20 claims prove nothing, so do not report it. For example, if almost every "supported" row got `incorrect`, a crop or the guide is probably wrong.

In [ ]:
mine = read_labels(f"{DRY}/annotator_dry.csv")
key_dry = pd.read_csv(f"{DRY}/key_private.csv")
table_dry, joined_dry = checker_confusion(key_dry, mine)
print(table_dry)
print(joined_dry[joined_dry["human"] == "region not visible"][["tile_id", "cell", "source", "verdict"]])

### Real round

Edit the quotas once the dry run is fine. Each quota is a source and a verdict, so every source and every verdict is covered. The sample is stratified, not random: report agreement per verdict and per source, not as one overall rate for the whole pool. Both annotators get the same sheet and label it on their own (copy `annotator_sheet.csv` to `annotator_A.csv` and `annotator_B.csv`, one per person, and fill the `label` column). Do not share the key file.

In [ ]:
QUOTAS = {("qwen3vl", "supported"): 4, ("qwen3vl", "contradicted"): 16, ("qwen3vl", "not_verifiable"): 8,
          ("evidence_ref", "supported"): 14, ("evidence_ref", "contradicted"): 14,
          ("rscc", "supported"): 12, ("rscc", "contradicted"): 10, ("rscc", "not_verifiable"): 3}   # about 80 claims, because about a third may be unjudgeable

real = sample_claims(pool, QUOTAS)
real_sheet = write_annotation_files(real, TILES, f"{HUMAN_DIR}/round1")
print(real.groupby(["source", "verdict"]).size())

### Annotator packages

Do not share the `human_check` folder: it sits in the shared project folder and holds `key_private.csv`, which shows the verdicts. Build one clean folder per annotator outside the project folder and share only that folder with that person.

In [ ]:
import shutil

SHARE = "/content/drive/MyDrive/human_check_share"       # outside the shared project folder


def make_annotator_package(round_dir, out_dir, name):
    """Sheet, guide and pictures for one annotator, without the key file."""
    dest = f"{out_dir}/{name}"
    os.makedirs(dest, exist_ok=True)
    shutil.copytree(f"{round_dir}/crops", f"{dest}/crops", dirs_exist_ok=True)
    shutil.copy(f"{round_dir}/GUIDE.txt", dest)
    shutil.copy(f"{round_dir}/annotator_sheet.csv", f"{dest}/annotator_{name}.csv")
    return dest


for name in ("A", "B"):
    print(make_annotator_package(f"{HUMAN_DIR}/round1", SHARE, name))

## Phase 2: after both annotators have labelled

The label functions are defined in the dry-run section above. Put the two finished files in `human_check/round1/` with the exact names `annotator_A.csv` and `annotator_B.csv`. Each has the same columns as `annotator_sheet.csv` with the `label` column filled using exactly the three labels in `LABELS`.

In [ ]:
def align_labels(path_a, path_b, labels=LABELS):
    """Labels from two annotator files on the claim_ids both contain, after checking the claim text matches."""
    da = pd.read_csv(path_a).set_index("claim_id")
    db = pd.read_csv(path_b).set_index("claim_id")
    common = da.index.intersection(db.index)
    print(f"A: {len(da)} rows | B: {len(db)} rows | in both: {len(common)}")
    print("only in A:", sorted(da.index.difference(db.index))[:10])
    print("only in B:", sorted(db.index.difference(da.index))[:10])
    mismatch = common[da.loc[common, "claim"].values != db.loc[common, "claim"].values]
    if len(mismatch):
        raise ValueError(f"{len(mismatch)} claim_ids have different claim text in A and B, "
                         f"first: {list(mismatch[:5])}. The files come from different samples.")
    a, b = da.loc[common, "label"], db.loc[common, "label"]
    bad = ~(a.isin(labels) & b.isin(labels))
    if bad.any():
        raise ValueError(f"missing or unknown labels on claim_ids: {list(a.index[bad][:5])}")
    return a, b

In [ ]:
ROUND = f"{HUMAN_DIR}/round1"

a, b = align_labels(f"{ROUND}/annotator_A.csv", f"{ROUND}/annotator_B.csv")
key = pd.read_csv(f"{ROUND}/key_private.csv")

print("claims labelled:", len(a))
print("raw agreement:", round(agreement_rate(a, b), 3))
print("Cohen's kappa:", round(cohen_kappa(a, b), 3))
print(pd.crosstab(a, b, rownames=["annotator A"], colnames=["annotator B"]))

In [ ]:
def wilson_ci(k, n, z=1.96):
    """Wilson score interval for k successes out of n."""
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z ** 2 / n
    centre = (p + z ** 2 / (2 * n)) / d
    half = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
    return (centre - half, centre + half)


def verdict_only_agreement(a, b):
    """Agreement and kappa on the claims where both annotators chose correct or incorrect (no 'region not visible')."""
    both = a.isin(LABELS[:2]) & b.isin(LABELS[:2])
    x, y = a[both], b[both]
    k = int((x == y).sum())
    lo, hi = wilson_ci(k, len(x))
    return {"n": len(x), "agree": k, "rate": round(k / len(x), 3), "ci95": (round(float(lo), 2), round(float(hi), 2)),
            "kappa": round(cohen_kappa(x, y, labels=LABELS[:2]), 3)}


k_all = int((a == b).sum())
lo, hi = wilson_ci(k_all, len(a))
print(f"three labels: {k_all} of {len(a)} agree = {k_all / len(a):.3f}  (95% CI {lo:.2f} to {hi:.2f}), kappa {cohen_kappa(a, b):.3f}")
print("correct or incorrect only:", verdict_only_agreement(a, b))
print("\nlabel counts per annotator")
print(pd.concat([a.value_counts().rename("A"), b.value_counts().rename("B")], axis=1).fillna(0).astype(int))

In [ ]:
cons = consensus_labels(a, b)
print("disagreements to resolve:", int((cons == "disagree").sum()))
print(key[key["claim_id"].isin(cons.index[cons == "disagree"])][["claim_id", "tile_id", "cell", "claim"]])

Resolve each disagreement by discussion (both annotators look at the crop together), then write the agreed label into a file `adjudicated.csv` with columns `claim_id,label` and rerun from the next cell. Skip this cell if there are no disagreements.

In [ ]:
adj_path = f"{ROUND}/adjudicated.csv"
if os.path.exists(adj_path):
    adj = pd.read_csv(adj_path).set_index("claim_id")["label"]
    cons = cons.copy()
    cons.loc[adj.index] = adj
remaining = int((cons == "disagree").sum())
print("unresolved disagreements:", remaining)
final = cons[cons != "disagree"]

In [ ]:
table, joined = checker_confusion(key[key["claim_id"].isin(final.index)], final)
print(table)

judged = joined[joined["human"].isin(["correct", "incorrect"]) & joined["verdict"].isin(["supported", "contradicted"])]
agree = ((judged["verdict"] == "supported") == (judged["human"] == "correct")).mean() if len(judged) else float("nan")
print("checker agrees with humans on the claims it could judge:", round(float(agree), 3), "of", len(judged))

In [ ]:
fig = plot_confusion(table)
fig.savefig(f"{ROUND}/checker_vs_human_confusion.png", dpi=150)

In [ ]:
def event_type(tile_id):
    """'flooding' for the flooding events (Nepal, Midwest), 'hurricane' for the hurricane events, else 'other'."""
    return "flooding" if "flooding" in tile_id else "hurricane" if "hurricane" in tile_id else "other"


def agreement_by(joined, column):
    """Checker-human agreement per group (supported = correct, contradicted = incorrect), with the number of claims judged.

    Only rows where the checker said supported or contradicted and the human said correct or incorrect are counted.
    """
    j = joined[joined["human"].isin(["correct", "incorrect"]) & joined["verdict"].isin(["supported", "contradicted"])].copy()
    j["match"] = ((j["verdict"] == "supported") == (j["human"] == "correct")).astype(float)
    return j.groupby(column)["match"].agg(["mean", "count"]).round(3)


def not_visible_rate(joined, column):
    """Share of claims per group that the human labelled 'region not visible', with the group size."""
    g = joined.assign(nv=(joined["human"] == "region not visible").astype(float)).groupby(column)["nv"]
    return pd.concat([g.mean().round(3).rename("not_visible"), g.size().rename("claims")], axis=1)


joined = joined.assign(event_type=joined["tile_id"].map(event_type))
print("agreement by source"); print(agreement_by(joined, "source"))
print("\nagreement by event type"); print(agreement_by(joined, "event_type"))
print("\nagreement by source and checker verdict"); print(agreement_by(joined, ["source", "verdict"]))
print("\nnot visible by source"); print(not_visible_rate(joined, "source"))
print("\nnot visible by event type"); print(not_visible_rate(joined, "event_type"))

### Phase 3: rule-based resolution of the low-end disagreements (single rater)

Annotator B was not available to adjudicate, so only the disagreements that a written rule can decide are resolved, and only by one person (annotator A).
These are the "none" and "few" claims. For each one the crop is shown **without** either annotator's label and without the checker's verdict. You count the buildings, then save the counts with `step(SHEET, claim=<id on screen>, total=<n_total>, claimed=<n_claimed>)`, which also shows the next row:
- `n_total`: buildings clearly visible in the crop (0 means no building is visible, so the label is `region not visible`).
- `n_claimed`: how many of them show the damage the statement names (any damage for "damaged", otherwise only the named level). Apply the flood rule as you did for A.

The label then follows from the bins in `checker.bin_of` (none = 0, few under 10%, some 10 to 30%, many 30 to 60%, most 60% and above).
The disagreements that are not low-end stay unresolved.

Report these rows apart from the rows where A and B agreed on their own: the counts come from a person, but the mapping from counts to a label is the checker's own rule, so agreement on these rows is partly circular. Call the labels "count-derived, single rater", not a consensus.

In [ ]:
from IPython.display import display, clear_output
from PIL import Image
from floodsitrep.nlp.checker import bin_of

LOW_END = ("none", "few")


def claimed_class(claim):
    """What to count for a claim: 'any damage', 'minor', 'major' or 'destroyed'."""
    for word in ("minor", "major", "destroyed"):
        if word in claim:
            return word
    return "any damage"


def disagreement_table(key, a, b):
    """The claims A and B labelled differently, with both labels, the quantity word and what to count."""
    d = key[key["claim_id"].isin(a.index[a != b])].copy()
    d["label_A"], d["label_B"] = d["claim_id"].map(a), d["claim_id"].map(b)
    d["quantity"] = d["claim"].str.extract(r"^(none|few|some|many|most)")[0]
    d["count_what"] = d["claim"].map(claimed_class)
    return d.reset_index(drop=True)


def make_count_sheet(dis, quantities=LOW_END):
    """Rows to count: the disagreements with a low-end quantity word, with empty n_total and n_claimed."""
    s = dis[dis["quantity"].isin(quantities)].copy()
    s["image"] = s["claim_id"].map(lambda i: f"crops/claim_{i:03d}.png")
    s["n_total"], s["n_claimed"] = "", ""
    return s[["claim_id", "image", "claim", "quantity", "count_what", "n_total", "n_claimed", "label_A", "label_B"]]


def step(sheet_path, claim=None, total=None, claimed=None):
    """Save the counts for the row on screen (the claim id must match), then show the next uncounted row.
    Call step(SHEET) to show the first uncounted row. The label columns are not shown, so the counts are made blind."""
    s = pd.read_csv(sheet_path, dtype=str).fillna("")
    todo = lambda: s[(s["n_total"] == "") | (s["n_claimed"] == "")]
    if total is not None:
        row = todo().iloc[0]
        assert str(claim) == row["claim_id"], f"the row on screen is claim {row['claim_id']}, not {claim}"
        assert 0 <= claimed <= total, "claimed must be between 0 and total"
        s.loc[row.name, ["n_total", "n_claimed"]] = str(total), str(claimed)
        s.to_csv(sheet_path, index=False)
    if todo().empty:
        print("all rows counted, saved to", sheet_path)
        return
    r = todo().iloc[0]
    clear_output(wait=True)
    print(f"{len(s) - len(todo()) + 1} of {len(s)}   (claim {r['claim_id']})")
    display(Image.open(f"{ROUND}/{r['image']}"))
    print("STATEMENT:", r["claim"])
    print("COUNT: buildings visible in total, then how many show:", r["count_what"])
    print(f"\nthen run:  step(SHEET, claim={r['claim_id']}, total=?, claimed=?)")


def label_from_counts(quantity, n_total, n_claimed):
    """region not visible when nothing is visible, else correct when the share falls in the claimed bin."""
    if n_total == 0:
        return "region not visible"
    return "correct" if bin_of(n_claimed / n_total) == quantity else "incorrect"


def resolve_from_counts(sheet):
    """Rows with both counts filled in, with their count-derived label (rows without counts are left out)."""
    s = sheet[(sheet["n_total"].astype(str) != "") & (sheet["n_claimed"].astype(str) != "")].copy()
    s["claim_id"] = s["claim_id"].astype(int)          # the sheet is read back as text
    s["final_label"] = [label_from_counts(q, int(t), int(c)) for q, t, c in zip(s["quantity"], s["n_total"], s["n_claimed"])]
    s["reason"] = "rule_count_single_rater"
    return s

Build the count sheet and count the buildings (the crop is shown for each row; the labels are not). Rerun `step(SHEET)` at any time to continue.

In [ ]:
dis = disagreement_table(key, a, b)
print(len(dis), "disagreements:", dis["quantity"].value_counts().to_dict())

import os, shutil
OLD = f"{ROUND}/count_sheet.csv"
OLD_COPY = f"{ROUND}/count_sheet_old_sample.csv"
if os.path.exists(OLD) and not os.path.exists(OLD_COPY):
    shutil.copy(OLD, OLD_COPY)                                # keep the old-sample counts, renamed

SHEET = f"{ROUND}/count_sheet_v2.csv"
if os.path.exists(SHEET):
    print("count_sheet_v2.csv already exists, keeping the counts in it")   # never overwrite saved counts
else:
    new_sheet = make_count_sheet(dis)
    new_sheet.to_csv(SHEET, index=False)
    print(len(new_sheet), new_sheet["claim_id"].tolist())     # expect 13: 1 6 10 15 17 29 30 43 59 63 70 71 72

Turn the counts into labels and compare them with A and B.

In [ ]:
resolved = resolve_from_counts(pd.read_csv(SHEET, dtype=str).fillna(""))
resolved.to_csv(f"{ROUND}/resolved_single_rater.csv", index=False)
print(len(resolved), "of", int(dis["quantity"].isin(LOW_END).sum()), "low-end disagreements resolved")
print(resolved["final_label"].value_counts().to_string())

res = resolved.set_index("claim_id")["final_label"]
print("equals A:", int((resolved["claim_id"].map(a) == resolved["final_label"]).sum()),
      "| equals B:", int((resolved["claim_id"].map(b) == resolved["final_label"]).sum()),
      "| equals neither:", int(((resolved["claim_id"].map(a) != resolved["final_label"]) &
                                  (resolved["claim_id"].map(b) != resolved["final_label"])).sum()))

In [ ]:
x = resolved.assign(A=resolved["claim_id"].map(a), B=resolved["claim_id"].map(b))
print(x[(x["final_label"] != x["A"]) | (x["final_label"] == "region not visible")]
      [["claim_id", "claim", "n_total", "n_claimed", "A", "B", "final_label"]].to_string())

Checker agreement, with the two kinds of human label kept apart.

In [ ]:
# Rows where A and B agreed (or that you adjudicated in the earlier cell) plus the count-derived rows, kept apart by basis
combined = pd.concat([final, res[~res.index.isin(final.index)]])
table_c, joined_c = checker_confusion(key[key["claim_id"].isin(combined.index)], combined)
joined_c = joined_c.assign(event_type=joined_c["tile_id"].map(event_type),
                           basis=np.where(joined_c.index.isin(res.index), "count rule (single rater)", "annotators agreed"))
print("agreement by basis"); print(agreement_by(joined_c, "basis"))
print("\nagreement by source and basis"); print(agreement_by(joined_c, ["source", "basis"]))
print("\nagreement by source, verdict and basis"); print(agreement_by(joined_c, ["source", "verdict", "basis"]))

In [ ]:
def with_wilson(table):
    """Add a 95% Wilson interval to an agreement_by table (columns mean and count)."""
    k = (table["mean"] * table["count"]).round().astype(int)
    ci = [wilson_ci(int(kk), int(n)) for kk, n in zip(k, table["count"])]
    return table.assign(ci_low=[round(c[0], 2) for c in ci], ci_high=[round(c[1], 2) for c in ci])


print("overall");            print(with_wilson(agreement_by(joined_c.assign(all_claims="all"), "all_claims")))
print("\nby basis");          print(with_wilson(agreement_by(joined_c, "basis")))
print("\nby source");         print(with_wilson(agreement_by(joined_c, "source")))
print("\nby event type");     print(with_wilson(agreement_by(joined_c, "event_type")))
print("\nby checker verdict"); print(with_wilson(agreement_by(joined_c, "verdict")))

In [ ]:
def false_contradictions(joined):
    """Claims the checker contradicted but the human label says are correct (the checker's false contradictions)."""
    d = joined[(joined["verdict"] == "contradicted") & (joined["human"] == "correct")]
    return d[["tile_id", "cell", "source", "claim", "basis"]]


fc = false_contradictions(joined_c)
print(len(fc), "false contradictions")
fc

In [ ]:
import json
from floodsitrep.nlp.checker import class_fraction

def cell_counts(tile_id, cell):
    """Buildings and damage counts in the usable cells the checker used for a claim."""
    ev = json.load(open(f"{EVIDENCE_DIR}/{tile_id}.json"))
    cells = ev["cells"] if cell == "all" else [c for c in ev["cells"] if c["cell"] == cell]
    cells = [c for c in cells if c["buildings"] > 0 and c.get("valid_fraction", 1.0) >= 0.5]
    return sum(c["buildings"] for c in cells), {k: round(sum(c["buildings"] * (class_fraction([c], k) or 0.0) for c in cells), 1)
                                                for k in ("damaged", "minor", "major", "destroyed")}

for cid, r in fc.iterrows():
    print(cid, r["tile_id"], r["cell"], r["claim"][:28], cell_counts(r["tile_id"], r["cell"]))

In [ ]:
from floodsitrep.nlp.checker import class_fraction

def tile_share(tile_id):
    """(buildings, damaged, share) over the usable cells of a tile, as tile_verdict computes it."""
    ev = json.load(open(f"{EVIDENCE_DIR}/{tile_id}.json"))
    usable = [c for c in ev["cells"] if c["buildings"] > 0 and c.get("valid_fraction", 1.0) >= 0.5]
    total = sum(c["buildings"] for c in usable)
    dam = sum(c["buildings"] * (class_fraction([c], "damaged") or 0.0) for c in usable)
    return total, round(dam), (round(dam / total, 3) if total else None)

rows = joined_c[(joined_c["source"] == "rscc") & (joined_c["verdict"] == "contradicted")]
for cid, r in rows.iterrows():
    print(cid, r["tile_id"], r["human"], r["basis"], tile_share(r["tile_id"]))

In [ ]:
ORDER = ["none", "few", "some", "many", "most"]

def inconsistent_pairs(joined):
    """Pairs of claims on the same tile and cell, both labelled correct, that cannot both be true."""
    d = joined[joined["human"] == "correct"].assign(
        q=lambda x: x["claim"].str.extract(r"^(none|few|some|many|most)")[0],
        cls=lambda x: x["claim"].map(claimed_class)).dropna(subset=["q"])
    out = []
    for (tile, cell), g in d.groupby(["tile_id", "cell"]):
        rows = list(g.itertuples())
        for i, a in enumerate(rows):
            for b in rows[i + 1:]:
                if a.cls == b.cls:
                    bad = a.q != b.q
                elif "any damage" in (a.cls, b.cls):
                    spec, anyd = (a, b) if b.cls == "any damage" else (b, a)
                    bad = ORDER.index(spec.q) > ORDER.index(anyd.q)
                else:
                    bad = False
                if bad:
                    out.append({"tile_id": tile, "cell": cell, "claim_a": a.Index, "text_a": a.claim,
                                "claim_b": b.Index, "text_b": b.claim})
    return pd.DataFrame(out)

inconsistent_pairs(joined_c)

In [ ]:
def tile_classes(tile_id):
    ev = json.load(open(f"{EVIDENCE_DIR}/{tile_id}.json"))
    usable = [c for c in ev["cells"] if c["buildings"] > 0 and c.get("valid_fraction", 1.0) >= 0.5]
    return {k: round(sum(c["buildings"] * (class_fraction([c], k) or 0.0) for c in usable))
            for k in ("minor", "major", "destroyed")}
for cid in (6, 10, 26, 37, 59):
    print(cid, tile_classes(joined_c.loc[cid, "tile_id"]))

## Notes for the report
- Report raw agreement and kappa between annotators, and how disagreements were resolved.
- Report the confusion table and the agreement of the checker with the humans on the claims it could judge. Give the sample sizes next to every rate: this is a small sample.
- `not_verifiable` claims should mostly be labelled `region not visible` or have no buildings; a `not_verifiable` claim that humans find clearly `correct` or `incorrect` shows a gap in what the evidence can cover.
- Annotators saw only the crops, the claim and the label guide, not the verdict or the source.
- Report results per source and, if the counts allow, per event type (flooding against hurricane events), with the number of claims judged next to each rate. Also report how often `region not visible` was used, per source.
- Report results per source. `rscc` claims are tile-level (extracted from RSCC captions with simple rules, about 1 in 5 captions); say so. `evidence_ref` claims are built from the evidence (not model output), so say so and do not present their rates as model quality.
- Phase 3: B could not adjudicate, so the low-end disagreements (none, few) were resolved by one person (A) from building counts; report them as count-derived labels from a single rater, apart from the rows where A and B agreed, and keep the initial kappa (before any resolution). The checker uses the same bins as the count rule, so agreement on those rows is partly circular.
- Report both kappas from the annotator comparison: all three labels, and correct or incorrect only (the second shows that most disagreement is about `region not visible`). Say which one is the headline, and give the Wilson interval next to each rate.
- Say who labelled each annotator file, which rule each applied (the guide above does not state the flood rule: a building standing in floodwater counts as damaged, even if the roof looks intact), and whether any labels were model-assisted. Fill this in before submitting; do not leave it implied.
- Keep the first round's agreement figures (kappa 0.162, 49% checker-human agreement) next to these. The sample and the annotators differ, so do not present the change as an effect of one cause.
- Checker-human agreement computed on annotator-agreed claims only leaves out the harder, disputed ones. Report it again after Phase 3, with the basis (annotators agreed, or count rule from one rater) shown separately.
- The sample is stratified by source and verdict, so rates are per class, not accuracy over all claims.